# 03 — Krigeage ordinaire (baseline géostatistique)

Ce notebook met en œuvre le **premier pipeline complet** de
reconstruction de la carte probabiliste :

1. Simuler un champ et placer 20 capteurs uniformes.
2. Estimer un variogramme empirique sur les observations binaires.
3. Krigeage ordinaire pour prédire ``p̂`` partout sur la grille.
4. Calculer les métriques (AUC, Brier, log-loss, MAE/RMSE) en comparant à
   la vérité terrain simulée.
5. Visualiser : carte vraie, carte prédite, carte d'erreur, carte
   d'incertitude.

In [ ]:
import logging
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from aphid_spatial.evaluation.metrics import calibration_curve_data, evaluate_all
from aphid_spatial.methods.geostatistics import OrdinaryKrigingIndicator
from aphid_spatial.simulation import (
    Field,
    FieldConfig,
    SensorConfig,
    place_sensors,
    simulate_field,
)
from aphid_spatial.visualization.maps import plot_summary

logging.basicConfig(level=logging.INFO, format="%(name)s | %(message)s")
FIG_DIR = Path("../outputs/figures")
RES_DIR = Path("../outputs/results")
FIG_DIR.mkdir(parents=True, exist_ok=True)
RES_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
data_path = Path("../data/simulated/field_default.npz")
if data_path.exists():
    field = Field.load(data_path)
else:
    field = simulate_field(FieldConfig(seed=42))

readings = place_sensors(
    field, SensorConfig(n_sensors=20, placement="uniform", seed=2024)
)
print(f"capteurs : {readings.coords.shape[0]} ; obs+ : {int(readings.obs.sum())}")

## Variogramme empirique sur les observations

Avec seulement 20 capteurs, le variogramme est instable. ``pykrige``
l'estime et l'ajuste automatiquement (modèle exponentiel par défaut).

In [ ]:
method = OrdinaryKrigingIndicator(variogram_model="exponential", n_lags=8)
method.fit(readings, field)
if method._kriger is not None:
    print("variogram_model_parameters :", method._kriger.variogram_model_parameters)
else:
    print("Fallback : prédiction constante")

In [ ]:
p_pred = method.predict_proba(field.coords)
sigma = method.predict_uncertainty(field.coords)

results = evaluate_all(field.presence, p_pred, p_true=field.prob)
for k, v in results.items():
    print(f"{k:>16} : {v:.4f}")

In [ ]:
fig = plot_summary(field, readings, p_pred, sigma=sigma)
fig.savefig(FIG_DIR / "03_geostatistics_summary.png", dpi=150)
plt.show()

## Courbe de calibration (reliability diagram)

In [ ]:
calib = calibration_curve_data(field.presence, p_pred, n_bins=10)

fig, ax = plt.subplots(figsize=(5, 5))
ax.plot([0, 1], [0, 1], "k--", alpha=0.6, label="parfait")
valid = calib["count"] > 0
ax.plot(calib["mean_pred"][valid], calib["frac_pos"][valid], "o-", label="OK indicator")
ax.set_xlabel("probabilité prédite moyenne")
ax.set_ylabel("fréquence empirique de présence")
ax.set_title("Calibration")
ax.legend()
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
fig.savefig(FIG_DIR / "03_geostatistics_calibration.png", dpi=150)
plt.show()

## Sauvegarde des métriques

In [ ]:
import pandas as pd

row = {"method": method.name, "placement": readings.config.placement, "n_sensors": readings.config.n_sensors, **results}
df = pd.DataFrame([row])
out_path = RES_DIR / "03_geostatistics_metrics.csv"
df.to_csv(out_path, index=False)
print(f"Métriques sauvegardées : {out_path}")
df